<a href="https://colab.research.google.com/github/WARRAICH-11/NETSOL/blob/main/Project_2_Student_Grade_Book.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction

This in-class project asks you to build a **Student Grade Book** — a self-contained system for managing student scores, computing averages, assigning letter grades, and printing a full class summary report.

This is a **standalone project**. There are no hints or step-by-step scaffolding. You are given a detailed problem statement, a description of what the final output should look like, and some ideas for how to approach it. The rest is up to you.

By the end of this project you should be comfortable with:

* Designing a class from scratch with **instance variables**, **methods**, and **properties**
* Using **dictionaries** and **lists** to store and retrieve structured data
* Applying **list comprehensions** and `map` / `filter` to process records
* Writing clean, readable output with f-strings

Please make sure to run <span style="color: red;">all cells</span> when done.


---
## Problem Statement

Build a `GradeBook` class that manages student scores across multiple assignments for a course.

### What it must do

**Data management:**
- Store a course name (e.g. `"Web Dev Batch 2025"`)
- Allow adding students by name
- Allow logging a score for any student on any named assignment
- A student can have scores on many different assignments

**Computation:**
- Compute the average score for any given student
- Assign a letter grade based on average: `A ≥ 90`, `B ≥ 80`, `C ≥ 70`, `D ≥ 60`, `F` below 60
- Compute the overall class average (average of all student averages)
- Identify the top-performing student (highest average)
- Identify any student who is at risk — average below 60

**Output:**
- A `report()` method that prints the full class summary (see below)
- A `__str__` that gives a one-line summary of the grade book
- A `__len__` that returns the number of students enrolled

### What a complete run should look like

```python
gb = GradeBook("Web Dev Batch 2025")
gb.add_student("Ali")
gb.add_student("Sara")
gb.add_student("Hamza")
gb.add_student("Ayesha")

gb.log_score("Ali",    "Assignment 1", 78)
gb.log_score("Ali",    "Assignment 2", 91)
gb.log_score("Ali",    "Quiz 1",       85)
gb.log_score("Sara",   "Assignment 1", 62)
gb.log_score("Sara",   "Assignment 2", 55)
gb.log_score("Sara",   "Quiz 1",       48)
gb.log_score("Hamza",  "Assignment 1", 95)
gb.log_score("Hamza",  "Assignment 2", 88)
gb.log_score("Hamza",  "Quiz 1",       92)
gb.log_score("Ayesha", "Assignment 1", 74)
gb.log_score("Ayesha", "Assignment 2", 80)
gb.log_score("Ayesha", "Quiz 1",       69)

print(gb)
print()
gb.report()
```

```
GradeBook: Web Dev Batch 2025 | 4 students | 3 assignments

=== Web Dev Batch 2025 — Grade Report ===

Student       Avg     Grade   Status
─────────────────────────────────────
Ali           84.67   B       OK
Sara          55.00   F       ⚠ At Risk
Hamza         91.67   A       OK
Ayesha        74.33   C       OK

─────────────────────────────────────
Class Average : 76.42
Top Student   : Hamza (91.67)
At Risk       : Sara
```


---
## Ideas for How to Go About It

You are not required to follow this approach — it is just one way to think about the problem.

**Think about your data structure first.** Before writing any code, ask yourself: how will you store each student's scores? A dictionary that maps student names to a list of scores is a natural fit — `{"Ali": [78, 91, 85], "Sara": [62, 55, 48]}`. When you add a score with `log_score()`, you just append to the right list.

**Build one method at a time and test it.** Write `add_student()` and test it. Write `log_score()` and test it. Write `average()` and test it. Don't try to write everything at once.

**The `average()` method** should take a student name and return the mean of their scores. A list comprehension or the built-in `sum()` and `len()` will get you there without importing anything.

**The `grade()` method** should take an average (a float) and return a letter. A chain of `if / elif / else` is perfectly fine here.

**The `top_student` property** can use Python's built-in `max()` function with a `key` argument. Think about what key you'd sort by.

**The `report()` method** is the most involved — it loops over all students, computes their average and grade, and prints a formatted table. f-strings with width specifiers like `f"{name:<14}"` will help you align columns cleanly.

**`__len__`** should return the number of students. `__str__` should return a short summary line.

**Bonus ideas if you finish early:**
- Add a `missing_assignments(student, total_assignments)` method that tells you how many assignments a student has not yet submitted
- Add a `class_distribution()` method that returns a dict of how many students got each letter grade: `{"A": 1, "B": 1, "C": 1, "F": 1}`
- Use `filter` to get all at-risk students instead of an `if` inside the loop


---
## Your Code


In [1]:
class GradeBook:
    def __init__(self, course_name):
        self.course_name = course_name
        self.students = {}
        self.assignments = set()

    def add_student(self, student_name):
        if student_name not in self.students:
            self.students[student_name] = {}

    def log_score(self, student_name, assignment_name, score):
        if student_name not in self.students:
            self.add_student(student_name)
        self.students[student_name][assignment_name] = score
        self.assignments.add(assignment_name)

    def average(self, student_name):
        scores = self.students.get(student_name, {}).values()
        return sum(scores) / len(scores) if scores else 0.0

    def grade(self, avg):
        if avg >= 90:
            return "A"
        elif avg >= 80:
            return "B"
        elif avg >= 70:
            return "C"
        elif avg >= 60:
            return "D"
        else:
            return "F"

    @property
    def class_average(self):
        if not self.students:
            return 0.0
        averages = [self.average(student) for student in self.students]
        return sum(averages) / len(averages)

    @property
    def top_student(self):
        if not self.students:
            return None
        return max(self.students.keys(), key=lambda student: self.average(student))

    @property
    def at_risk_students(self):
        return [student for student in self.students if self.average(student) < 60]

    def report(self):
        print(f"=== {self.course_name} — Grade Report ===")
        print(f"{'Student':<14}{'Avg':<8}{'Grade':<8}{'Status'}")
        print("─" * 37)

        for student in self.students:
            avg = self.average(student)
            letter_grade = self.grade(avg)
            status = "⚠ At Risk" if avg < 60 else "OK"
            print(f"{student:<14}{avg:<8.2f}{letter_grade:<8}{status}")

        print("─" * 37)
        print(f"Class Average : {self.class_average:.2f}")

        top = self.top_student
        if top:
            print(f"Top Student   : {top} ({self.average(top):.2f})")

        at_risk = ", ".join(self.at_risk_students)
        print(f"At Risk       : {at_risk if at_risk else 'None'}")

    def __len__(self):
        return len(self.students)

    def __str__(self):
        return f"GradeBook: {self.course_name} | {len(self)} students | {len(self.assignments)} assignments"

---
## Test Your Implementation

Run the full test below. If your output matches the expected report above, you are done.


In [3]:
gb = GradeBook("Web Dev Batch 2025")
gb.add_student("Ali")
gb.add_student("Sara")
gb.add_student("Hamza")
gb.add_student("Ayesha")

gb.log_score("Ali",    "Assignment 1", 78)
gb.log_score("Ali",    "Assignment 2", 91)
gb.log_score("Ali",    "Quiz 1",       85)
gb.log_score("Sara",   "Assignment 1", 62)
gb.log_score("Sara",   "Assignment 2", 55)
gb.log_score("Sara",   "Quiz 1",       48)
gb.log_score("Hamza",  "Assignment 1", 95)
gb.log_score("Hamza",  "Assignment 2", 88)
gb.log_score("Hamza",  "Quiz 1",       92)
gb.log_score("Ayesha", "Assignment 1", 74)
gb.log_score("Ayesha", "Assignment 2", 80)
gb.log_score("Ayesha", "Quiz 1",       69)

print(gb)
print()
gb.report()


GradeBook: Web Dev Batch 2025 | 4 students | 3 assignments

=== Web Dev Batch 2025 — Grade Report ===
Student       Avg     Grade   Status
─────────────────────────────────────
Ali           84.67   B       OK
Sara          55.00   F       ⚠ At Risk
Hamza         91.67   A       OK
Ayesha        74.33   C       OK
─────────────────────────────────────
Class Average : 76.42
Top Student   : Hamza (91.67)
At Risk       : Sara
